# 03 · Indexing and broadcasting real data / Indexación y broadcasting con datos reales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/03-indexing-and-broadcasting.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed">PART III · EXERCISE · 15 MIN</span>

## Practise today / Practica hoy

Predict broadcasting shapes and standardize pixel columns safely when variance is zero.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Predecir formas de broadcasting y estandarizar columnas de píxeles de forma segura si la varianza es cero.</div></div>

## Explore later / Explora después

Select observations with named features, fancy indexing, and Boolean masks.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Seleccionar observaciones con nombres de características, índices avanzados y máscaras booleanas.</div></div>

Follow the core block immediately below. / Sigue el bloque esencial de abajo.

<!-- CORE-PATH -->
## Core path / Ruta esencial

Read and run this block from top to bottom: **recall → example → attempt → feedback → checkpoint**. Preparation and feedback definitions appear where needed. Try before opening a folded solution. Stop at **Core complete**; everything after it is **Explore later**.

🇪🇸 Lee y ejecuta este bloque de arriba abajo: **recuerda → ejemplo → intento → retroalimentación → comprobación**. La preparación y las funciones de comprobación aparecen donde se necesitan. Inténtalo antes de abrir una solución plegada. Detente en **Fin de la ruta esencial**; después empieza **Explora después**.

### Recall / Recuerda

Recall notebook 02: in a matrix of samples × features, which axis may be shuffled if labels move with it?

🇪🇸 Recuerda el cuaderno 02: en una matriz de muestras × características, ¿qué eje puedes barajar si mueves también las etiquetas?

## Setup / Preparación

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Two real datasets, both packaged with scikit-learn:

- breast-cancer measurements, for the indexing exercises;
- handwritten digits, for broadcasting and zero variance.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">mediciones de cáncer de mama, para los ejercicios de indexación;</li><li style="margin:.35em 0">dígitos manuscritos, para broadcasting y varianza cero.</li></ul></div>

### Core prep 1/2 · Preparación esencial

Run the next cell. / Ejecuta la siguiente celda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from sklearn.datasets import load_breast_cancer, load_digits

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

bc = load_breast_cancer()
X = bc.data
y = bc.target              # 0 = malignant, 1 = benign
names = list(bc.feature_names)

digits = load_digits()
images = digits.images     # (1797, 8, 8)

print("Breast-cancer matrix / Matriz de cáncer de mama:", X.shape)
print("Number of feature names / Número de características:", len(names))
print("Digit images / Imágenes de dígitos:", images.shape)
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

### Core prep 2/2 · Preparación esencial

Run the next cell. / Ejecuta la siguiente celda.

In [ ]:
D = images.reshape(len(images), -1)

print("Original images / Imágenes originales:", images.shape)
print("Flattened matrix / Matriz aplanada:", D.shape)
print("EN: each row is one image; each column is one pixel position.")
print("ES: cada fila es una imagen; cada columna es una posición de píxel.")

## 3.3 Broadcasting on real images / Broadcasting sobre imágenes reales

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

From selecting data to transforming it.

Each digit is an `8 × 8` image. Flattened, each becomes a row of 64 pixel
features: <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(124,58,237,.14);border:1px solid rgba(124,58,237,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(1797, 8, 8) → (1797, 64)</span>.

We want to standardize all 64 pixel columns.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">THE ANALOGY · LA ANALOGÍA</div>One mean per column gives <code>mean.shape = (64,)</code>. NumPy subtracts those 64 numbers from <b>every one</b> of the 1,797 rows, as if the row had been copied 1,797 times — without copying it.<div style='margin-top:10px'>That is broadcasting.</div></div>

The analogy says what broadcasting *does*. It does not say when it will
work — and that is decidable before you run anything, by three steps in
order.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">THE RULE · LA REGLA</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>Line the two shapes up <b>from the right</b>.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>Each facing pair must be <b>equal</b>, or one of them must be <b>1</b>.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span>A missing axis on the left counts as <b>1</b>.</div><div style='margin-top:10px'>Every pair passes, or NumPy raises. Nothing else is consulted — not what the axes mean, not which array is bigger.</div></div>

Apply it to the case above. `(1797, 64)` against `(64,)`, lined up from
the right: `64` faces `64`, a match; `1797` faces nothing at all, and by
step 3 that empty slot counts as `1`, so the pair is `1797` against `1`.
Every pair passes, and the result is `(1797, 64)`.

Now one that fails. One offset per *sample* would be `(1797,)`, and from
the right that is `1797` facing `64` — not equal, and neither is `1`.
NumPy raises. `[:, None]` makes it `(1797, 1)`, and `1` facing `64`
passes. That is the trap the second animation below draws.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una media por columna da <code>(64,)</code>. NumPy resta esos 64 números a <b>cada una</b> de las 1.797 filas, como si el vector se hubiera copiado 1.797 veces — sin copiarlo. Eso es broadcasting.<div style='margin-top:10px'><b>La regla</b>, en tres pasos: <b>1 ·</b> alinea las dos formas <b>por la derecha</b>; <b>2 ·</b> cada par enfrentado debe ser <b>igual</b>, o uno de los dos debe ser <b>1</b>; <b>3 ·</b> un eje que falta a la izquierda cuenta como <b>1</b>. Si algún par no pasa, NumPy lanza un error.</div><div style='margin-top:10px'>Aplícala: <code>(1797, 64)</code> con <code>(64,)</code> da <code>64</code> contra <code>64</code>, y <code>1797</code> sin nada enfrente: por el paso 3 ese hueco vacío cuenta como <code>1</code>, así que el par es <code>1797</code> contra <code>1</code>. Sale <code>(1797, 64)</code>. Un desplazamiento por muestra sería <code>(1797,)</code>, y por la derecha eso es <code>1797</code> contra <code>64</code>: ni iguales ni <code>1</code>, así que falla. <code>[:, None]</code> lo convierte en <code>(1797, 1)</code>, y <code>1</code> contra <code>64</code> sí pasa.</div></div>

### Two samples by hand / Dos muestras a mano

For `M = [[2, 10], [4, 14]]` (samples × features), the feature means are `[3, 12]`.
Subtracting this `(2,)` vector from both rows gives `[[-1, -2], [1, 2]]`:
the last axis matches features; the missing leading axis repeats over samples.
The digit matrix below applies this same rule to 64 pixel columns.

🇪🇸 Para `M = [[2, 10], [4, 14]]` (muestras × características), las medias por característica son `[3, 12]`.
Restar este vector `(2,)` de ambas filas da `[[-1, -2], [1, 2]]`:
el último eje coincide con las características; el eje inicial ausente se repite sobre las muestras.
La matriz de dígitos aplica esta regla a 64 columnas de píxeles.


## Exercise 3 — standardize, then find the trap / Ejercicio 3 — estandariza y encuentra la trampa

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

The usual formula is `Z = (D - mean) / std`.

$$
\mu_j = \frac{1}{n}\sum_{i=1}^{n} D_{ij}
\qquad
\sigma_j = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\bigl(D_{ij}-\mu_j\bigr)^{2}}
\qquad
Z_{ij} = \frac{D_{ij}-\mu_j}{\sigma_j}
$$

Read it as: $i$ walks down the 1,797 images and $j$ across the 64 pixel
columns. Both $\mu$ and $\sigma$ are computed **down** a column, so each is a
vector of 64 numbers, and the subtraction reuses it across every row. The trap
is the last term: if a pixel never changes then $\sigma_j = 0$, and the
division is by zero.


Real data has a surprise waiting in it.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">PREDICT FIRST · PREDICE PRIMERO</div>One pixel column has <code>std = 0</code>. What happens when the formula divides by it?</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La fórmula habitual es <code>Z = (D - mean) / std</code>, pero los datos reales guardan una sorpresa: ¿qué ocurre si una columna de píxeles tiene <code>std = 0</code> y dividimos por ese valor?</div>

### Optional hints / Pistas opcionales

Try first; open one hint at a time. / Inténtalo primero; abre una pista a la vez.

<details>
<summary>Hint 1 / Pista 1</summary>

One mean belongs to each pixel position. Which axis must disappear to leave one number per column?

🇪🇸 Cada posición de píxel tiene una media. ¿Qué eje debe desaparecer para dejar un número por columna?

</details>

<details>
<summary>Hint 2 / Pista 2</summary>

Use axis=0 for column statistics. Replace only zero standard deviations with 1 using np.where; the centered constant columns are already zero.

🇪🇸 Usa axis=0 para estadísticas por columna. Sustituye solo las desviaciones cero por 1 con np.where; las columnas constantes centradas ya valen cero.

</details>



In [ ]:
# Feedback helper / Función de comprobación — run before your attempt / ejecuta antes del intento
import numpy as np

def check_core_answer(D, mean, std, Z):
    """Check your column statistics and safe standardization / Comprueba tus resultados."""
    D, mean, std, Z = map(np.asarray, (D, mean, std, Z))
    assert D.ndim == 2, "Input must be samples × features / La entrada debe ser muestras × características."
    assert mean.shape == std.shape == (D.shape[1],), "One statistic per column: check axis / Una estadística por columna: revisa el eje."
    assert Z.shape == D.shape, "Keep samples and pixels in place / Conserva las muestras y los píxeles."
    assert all(np.isfinite(a).all() for a in (mean, std, Z)), "Nonfinite values: inspect zero denominators / Valores no finitos: revisa denominadores cero."
    assert np.allclose(mean, D.mean(axis=0)), "Means should describe pixel columns / Las medias deben describir columnas de píxeles."
    assert np.allclose(std, D.std(axis=0)), "Use population std for each pixel column / Usa la std poblacional por columna."
    expected = (D - D.mean(axis=0)) / np.where(D.std(axis=0) == 0, 1, D.std(axis=0))
    assert np.allclose(Z, expected), "Check centering, scaling, and sample order / Revisa centrado, escala y orden de muestras."
    return "Checks passed; explain the sample axis / Comprobaciones superadas; explica el eje de muestras."


### Core activity · Actividad esencial

**Predict → Run → Explain → Check**

1. **Predict.** What shapes should mean and std have? What if a pixel never changes?
2. **Run.** Complete Exercise 3.
3. **Explain.** Explain the role of the sample axis in this calculation.
4. **Check.** Run `check_core_answer(D, mean, std, Z)` on your own results before opening the solution. Check all results are finite. Nonconstant columns should have mean near 0 and std near 1; constant columns should be 0.

<details>
<summary>Español · Predice → Ejecuta → Explica → Comprueba</summary>

1. **Predice.** ¿Qué formas deben tener mean y std? ¿Y si un píxel nunca cambia?
2. **Ejecuta.** Completa el Ejercicio 3.
3. **Explica.** Explica el papel del eje de muestras en este cálculo.
4. **Comprueba.** Ejecuta `check_core_answer(D, mean, std, Z)` con tus resultados antes de abrir la solución. Comprueba valores finitos. Las columnas variables deben tener media cercana a 0 y std cercana a 1; las constantes deben quedar en 0.

</details>

Prediction / Predicción: ___  
Evidence / Evidencia: ___  
Revised explanation / Explicación revisada: ___

In [ ]:
# TODO 4 / TAREA 4
#
# EN:
# 1. Compute the mean and std of each of the 64 pixel columns.
# 2. Print mean.shape and std.shape.
# 3. Try Z_bad = (D - mean) / std.
# 4. Check whether Z_bad contains NaN.
#
# ES:
# 1. Calcula la media y std de cada una de las 64 columnas de píxeles.
# 2. Imprime mean.shape y std.shape.
# 3. Prueba Z_bad = (D - mean) / std.
# 4. Comprueba si Z_bad contiene NaN.
#
# TODO 5 / TAREA 5
#
# EN:
# 1. Count how many pixels have std == 0.
# 2. Explain why a real handwritten-digit dataset could contain such pixels.
# 3. Replace zero denominators safely and verify that NaN disappears.
#
# ES:
# 1. Cuenta cuántos píxeles tienen std == 0.
# 2. Explica por qué un conjunto real de dígitos manuscritos puede contener esos píxeles.
# 3. Sustituye de forma segura los denominadores cero y verifica que desaparezcan los NaN.
# Check your own results before opening the solution / Comprueba tus resultados antes de abrir la solución:
# check_core_answer(D, mean, std, Z)


In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

mean = D.mean(axis=0)
std = D.std(axis=0)

print("mean.shape / Forma de mean:", mean.shape)
print("std.shape / Forma de std:", std.shape)

with np.errstate(divide="ignore", invalid="ignore"):
    Z_bad = (D - mean) / std

print("NaN before fix / NaN antes de corregir:", bool(np.isnan(Z_bad).any()))

zero_variance = (std == 0)
zero_count = int(zero_variance.sum())

print("Zero-variance pixels / Píxeles de varianza cero:", zero_count)

safe_std = np.where(zero_variance, 1.0, std)
Z = (D - mean) / safe_std

print("NaN after fix / NaN después de corregir:", bool(np.isnan(Z).any()))
print()
print("EN: zero-variance pixels never change across the 1,797 images.")
print("ES: los píxeles de varianza cero nunca cambian entre las 1.797 imágenes.")
print("EN: dividing by zero created invalid values; the safe denominator prevents that.")
print("ES: dividir entre cero creó valores inválidos; el denominador seguro lo evita.")

fig, ax = plt.subplots(figsize=(3.6, 3.6))
ax.imshow(D.mean(axis=0).reshape(8, 8), cmap="gray")

zero_rows, zero_cols = np.where(zero_variance.reshape(8, 8))
ax.scatter(zero_cols, zero_rows, s=220, marker="s", facecolors="none")
ax.set_title("Zero-variance pixels / Píxeles de varianza cero")
ax.axis("off")

plt.tight_layout()
plt.show()
print(check_core_answer(D, mean, std, Z))


<details>
<summary><strong>Why did NaN appear? / ¿Por qué apareció NaN?</strong></summary>

Standardization divides by the standard deviation. A pixel that never changes
across every image has `std = 0`, so the formula asks NumPy to divide by zero,
and `NaN` is what comes back.

**The code revealed a property of the data.** A zero-variance feature is not
automatically a bug in your program.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La estandarización divide por la desviación estándar. Un píxel que nunca cambia tiene <code>std = 0</code>, la fórmula divide entre cero y sale <code>NaN</code>.<br><br><b>El código reveló una propiedad de los datos.</b> Una característica de varianza cero no es automáticamente un error de programación.</div>

</details>

### Checkpoint / Comprobación

**Independent checkpoint · 1 minute within Explain/Check.** After checking your code, close the hints and solution. Answer alone before comparing with a partner. A new matrix `X` has shape `(3, 2)` (samples × features), and each sample has an offset in `b`, shape `(3,)`. Does `X - b` work? Write a corrected expression, its output shape, and which axis repeats.

**Comprobación independiente · 1 minuto dentro de Explica/Comprueba.** Tras comprobar tu código, cierra las pistas y la solución. Responde individualmente antes de comparar con otra persona. Una nueva matriz `X` tiene forma `(3, 2)` (muestras × características) y cada muestra tiene un desplazamiento en `b`, forma `(3,)`. ¿Funciona `X - b`? Escribe una expresión corregida, su forma de salida y qué eje se repite.

<details>
<summary>Checkpoint answer — attempt first / Respuesta — inténtalo primero</summary>

`X - b` fails: its trailing dimensions 2 and 3 do not match. `X - b[:, None]` has shape `(3, 2)`; each sample offset repeats across feature columns.

🇪🇸 `X - b` falla: las dimensiones finales 2 y 3 no coinciden. `X - b[:, None]` tiene forma `(3, 2)`; el desplazamiento de cada muestra se repite sobre las columnas de características.

</details>


## Core complete / Fin de la ruta esencial

Keep your prediction, evidence, and explanation. Follow the facilitator’s quiz and break schedule before continuing.

🇪🇸 Guarda tu predicción, evidencia y explicación. Sigue las pausas y quizzes del facilitador antes de continuar.

[Next: Notebook 04 / Siguiente: cuaderno 04](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/04-reshape-and-transpose.ipynb).

## Explore later / Explora después

Optional reference, exercises, and explorers. These are outside this section’s live core. Continue in order when studying them; some reuse earlier setup.

🇪🇸 Material de consulta, ejercicios y exploradores opcionales. Quedan fuera de la ruta esencial en vivo. Continúa en orden al estudiarlos; algunos reutilizan la preparación anterior.

## Four ideas before we code / Cuatro ideas antes de programar

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Picture a spreadsheet.

| Concept / Concepto | Simple idea / Idea sencilla | In a spreadsheet / En una hoja de cálculo |
|---|---|---|
| **Indexing / Indexación** | Choose a position / Elegir una posición | “Give me column B” / “Dame la columna B” |
| **Fancy indexing** | Choose several at once / Elegir varias a la vez | “Give me rows 3, 8 and 20” / “Dame las filas 3, 8 y 20” |
| **Boolean mask / Máscara booleana** | Keep rows where a condition is `True` / Conservar filas donde una condición es `True` | “Only trips after 6 p.m.” / “Solo viajes después de las 6 p. m.” |
| **Broadcasting** | Reuse a small array across many rows / Reutilizar un arreglo pequeño en muchas filas | The same column rule on every row / La misma regla en cada fila |

One more word arrives later. **Standardization** subtracts a feature's mean and
divides by its standard deviation, so features become comparable.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una palabra más para después: <b>estandarización</b> es restar la media de una característica y dividir por su desviación estándar, para que las características sean comparables.</div>

## The mistake that does not crash / El error que no hace fallar el programa

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">WHY WE INDEX BY NAME · POR QUÉ INDEXAMOS POR NOMBRE</div>Pick the wrong numerical column and Python runs perfectly. You simply analyse the wrong measurement.</div>

That is why this notebook says `names.index("mean radius")` and not `X[:, 0]`
wherever the name is the thing actually meant.

The data is numerical measurements computed from digitized fine-needle
aspirate images. We use it to learn data operations, not to build a diagnostic
rule.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Si eliges por accidente otra columna numérica, Python sigue sin error: solo analizas la medición equivocada. Por eso indexamos por nombre, <code>names.index(&quot;mean radius&quot;)</code>, y no por <code>X[:, 0]</code>.<br><br>Son mediciones calculadas a partir de imágenes digitalizadas de aspiración con aguja fina. Las usamos para aprender operaciones con datos, no para construir una regla de diagnóstico.</div>

## 3.1 Indexing by name / Indexación por nombre

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Thirty measurement columns. Two ways to ask for one.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">SAY WHICH · DI CUÁL</div><div style="margin:.55em 0">❌ “Give me column 0.” / «Dame la columna 0.»</div><div style="margin:.55em 0">✅ “Give me the column called <code>mean radius</code>.” / «Dame la columna llamada <code>mean radius</code>.»</div></div>

The second says what you meant, and survives a change in column order.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La segunda expresa la intención y sobrevive a un cambio en el orden de las columnas.</div>

### The table underneath / La tabla que hay debajo

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

`X` is a `(569, 30)` block of floats and nothing more. Here are five of its rows and six of its columns with the header put back on, which is the last time you will see one: NumPy does not carry column names, so `names` is a separate Python list and `names.index(...)` is what does the job a column name would.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>X</code> es un bloque de decimales <code>(569, 30)</code> y nada más. Aquí hay cinco de sus filas y seis de sus columnas con el encabezado puesto de nuevo, que es la última vez que verás uno: NumPy no lleva nombres de columna, así que <code>names</code> es una lista de Python aparte y <code>names.index(...)</code> es lo que hace el trabajo que haría un nombre de columna.</div>

In [ ]:
# pandas appears here and nowhere else in this notebook: it is putting the
# header back on for one look, not becoming the way we hold the data.
import pandas as pd

head = pd.DataFrame(X[:5, :6], columns=names[:6])
display(head.round(2))

print("Full matrix / Matriz completa:", X.shape)
# str(), because bc.feature_names is a NumPy array and its entries
# repr as np.str_(...), which is noise in a teaching print.
print("Columns shown / Columnas mostradas:",
      ", ".join(str(n) for n in names[:6]))
print()
print("EN: the header is not in X — it lives in the separate list `names`.")
print("ES: el encabezado no está en X: vive en la lista aparte `names`.")
print(
    "mean radius is column / mean radius es la columna:",
    names.index("mean radius"),
)


## Exercise 1 — indexing by name / Ejercicio 1 — indexación por nombre

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Extract one real measurement for all 569 samples.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">PREDICT FIRST · PREDICE PRIMERO</div>If <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(124,58,237,.14);border:1px solid rgba(124,58,237,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">X.shape == (569, 30)</span> and you select one column, what shape comes back?</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Extrae una medición real para las 569 muestras. Si <code>X.shape == (569, 30)</code> y seleccionas una columna, ¿qué forma debería salir?</div>

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Print X.shape and explain what each axis counts.
# 2. Find the position of "mean radius" with names.index(...).
# 3. Extract that column for all samples.
# 4. Print its shape.
#
# ES:
# 1. Imprime X.shape y explica qué cuenta cada eje.
# 2. Encuentra la posición de "mean radius" con names.index(...).
# 3. Extrae esa columna para todas las muestras.
# 4. Imprime su forma.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

print("X.shape / Forma de X:", X.shape)
print("EN: 569 samples × 30 measured features.")
print("ES: 569 muestras × 30 características medidas.")
print()

radius_idx = names.index("mean radius")
radius = X[:, radius_idx]

print("Column name / Nombre de columna:", names[radius_idx])
print("Column position / Posición de columna:", radius_idx)
print("radius.shape / Forma:", radius.shape)
print()
print("EN: selecting one column removes the feature axis from the result.")
print("ES: seleccionar una sola columna elimina el eje de características del resultado.")

### Feature explorer / Explorador de características

Pick any of the 30 feature names. You get its column position, its range, its
mean, and a histogram of the real measurements.

Indexing by name stops being abstract.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige cualquiera de las 30 características: posición de columna, rango, media y un histograma de las mediciones reales. La indexación por nombre deja de ser abstracta.</div>

In [ ]:
#@title 🔍 Feature explorer / Explorador de características — run me / ejecútame { display-mode: 'form' }

feature_dropdown = widgets.Dropdown(
    options=names,
    value="mean radius",
    description="Feature / Característica:",
    style={"description_width": "150px"},
)

def explore_feature(feature_name):
    i = names.index(feature_name)
    col = X[:, i]

    plt.close("all")
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    ax.hist(col, bins=30)
    ax.set_title(f"{feature_name} — real measurements / mediciones reales")
    ax.set_xlabel("value / valor")
    ax.set_ylabel("count / cantidad")
    plt.tight_layout()
    plt.show()

    print("Column / Columna:", i)
    print("Shape / Forma:", col.shape)
    print(f"Mean / Media: {col.mean():.3f}")
    print(f"Min / Mínimo: {col.min():.3f}")
    print(f"Max / Máximo: {col.max():.3f}")
    print("EN: one named feature has been selected for all 569 samples.")
    print("ES: se seleccionó una característica por nombre para las 569 muestras.")

feature_output = widgets.interactive_output(
    explore_feature,
    {"feature_name": feature_dropdown},
)

display(widgets.VBox([feature_dropdown, feature_output]))

<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

`X` has two axes: `(samples, features)`.

In `X[:, radius_idx]`:

- `:` keeps **all samples**;
- `radius_idx` keeps **one feature**.

One value per sample, so `(569,)`.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>X</code> tiene dos ejes: <code>(muestras, características)</code>. En <code>X[:, radius_idx]</code>, <code>:</code> conserva todas las muestras y <code>radius_idx</code> conserva una característica. Un valor por muestra: <code>(569,)</code>.</div>

</details>

## 3.2 Fancy indexing and Boolean masks / Fancy indexing y máscaras booleanas

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Two ideas, two different selection problems.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">WHICH ONE · CUÁL DE LAS DOS</div><div style="margin:.55em 0"><b>Fancy indexing</b> — you already know the positions. <code>X[[2, 10, 50], :]</code> is “rows 2, 10 and 50”.</div><div style="margin:.55em 0"><b>Boolean mask</b> — a condition defines the rows. <code>X[y == 0]</code> is “every row where <code>y == 0</code>”.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>Fancy indexing:</b> cuando ya conoces las posiciones. <b>Máscara booleana:</b> cuando una condición define las filas.</div>

## Exercise 2 — select observations / Ejercicio 2 — selecciona observaciones

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Two questions.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">ANSWER BOTH · RESPONDE LAS DOS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>Which five samples have the largest <code>mean radius</code>?</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>How does <code>mean radius</code> differ on average, in this dataset, between the two target groups?</div></div>

The second answer is descriptive. It is not a one-feature diagnostic rule.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>1 ·</b> ¿qué cinco muestras tienen el <code>mean radius</code> más alto? <b>2 ·</b> ¿cómo difiere en promedio entre los dos grupos, dentro de este conjunto?<br><br>La segunda respuesta es descriptiva: no es una regla de diagnóstico de una sola característica.</div>

In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Find the indices of the 5 largest values in `radius`.
# 2. Use fancy indexing to extract their full 30-feature profiles.
# 3. Verify that the result has shape (5, 30).
#
# ES:
# 1. Encuentra los índices de los 5 valores más grandes de `radius`.
# 2. Usa fancy indexing para extraer sus perfiles completos de 30 características.
# 3. Verifica que el resultado tenga forma (5, 30).
#
# TODO 3 / TAREA 3
#
# EN:
# Use Boolean masks to compute the mean radius for:
# - y == 0 (malignant)
# - y == 1 (benign)
#
# ES:
# Usa máscaras booleanas para calcular el radio medio para:
# - y == 0 (maligno)
# - y == 1 (benigno)

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

top5 = np.argsort(radius)[-5:]
profiles = X[top5, :]

print("Top-5 row indices / Índices de las 5 filas:", top5.tolist())
print("Profiles shape / Forma de perfiles:", profiles.shape)
print()

malignant_mask = (y == 0)
benign_mask = (y == 1)

malignant_mean = radius[malignant_mask].mean()
benign_mean = radius[benign_mask].mean()

print(f"Malignant mean radius / Radio medio maligno: {malignant_mean:.3f}")
print(f"Benign mean radius / Radio medio benigno: {benign_mean:.3f}")
print()
print("EN: in this dataset, the malignant group has a larger mean radius on average.")
print("ES: en este conjunto, el grupo maligno tiene un radio medio mayor en promedio.")
print("EN: this is a dataset description, not a diagnostic threshold.")
print("ES: esto describe el conjunto de datos; no es un umbral diagnóstico.")

### Mask explorer / Explorador de máscaras

Pick a group and a feature. The mask decides **which rows stay**; the feature
selector decides **which column you look at**.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un grupo y una característica: la máscara decide <b>qué filas quedan</b> y el selector decide <b>qué columna miras</b>.</div>

In [ ]:
#@title 🎭 Mask explorer / Explorador de máscaras — run me / ejecútame { display-mode: 'form' }

group_toggle = widgets.ToggleButtons(
    options=[
        ("All / Todas", "all"),
        ("Malignant / Malignas", "malignant"),
        ("Benign / Benignas", "benign"),
    ],
    value="all",
    description="Rows / Filas:",
    style={"description_width": "100px"},
)

mask_feature = widgets.Dropdown(
    options=names,
    value="mean radius",
    description="Feature / Característica:",
    style={"description_width": "150px"},
)

def explore_mask(group, feature_name):
    i = names.index(feature_name)

    if group == "malignant":
        mask = (y == 0)
        group_en = "malignant"
        group_es = "maligno"
    elif group == "benign":
        mask = (y == 1)
        group_en = "benign"
        group_es = "benigno"
    else:
        mask = np.ones(len(y), dtype=bool)
        group_en = "all samples"
        group_es = "todas las muestras"

    selected = X[mask, i]

    plt.close("all")
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    ax.hist(selected, bins=30)
    ax.set_title(f"{feature_name} — {group_en} / {group_es}")
    ax.set_xlabel("value / valor")
    ax.set_ylabel("count / cantidad")
    plt.tight_layout()
    plt.show()

    print("True values in mask / Valores True en la máscara:", int(mask.sum()))
    print("Selected shape / Forma seleccionada:", selected.shape)
    print(f"Mean / Media: {selected.mean():.3f}")
    print("EN: True means 'keep this row'.")
    print("ES: True significa 'conservar esta fila'.")

mask_output = widgets.interactive_output(
    explore_mask,
    {"group": group_toggle, "feature_name": mask_feature},
)

display(widgets.VBox([group_toggle, mask_feature, mask_output]))

### Compare all 30 features / Compara las 30 características

`mean radius` is one measurement out of thirty. Walk the list and compare the
two group distributions. Some separate visibly. Most do not.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>mean radius</code> es una de treinta. Recorre la lista y compara las distribuciones de ambos grupos: algunas se separan a la vista, la mayoría no.</div>

In [ ]:
#@title 📊 Feature comparison / Comparación de características — run me / ejecútame { display-mode: 'form' }

comparison_feature = widgets.Dropdown(
    options=names,
    value="mean radius",
    description="Feature / Característica:",
    style={"description_width": "150px"},
)

def compare_groups(feature_name):
    i = names.index(feature_name)
    col = X[:, i]

    plt.close("all")
    fig, ax = plt.subplots(figsize=(6.8, 3.4))
    ax.hist(col[y == 0], bins=30, alpha=0.6, label="malignant / maligno")
    ax.hist(col[y == 1], bins=30, alpha=0.6, label="benign / benigno")
    ax.set_title(feature_name)
    ax.set_xlabel("value / valor")
    ax.set_ylabel("count / cantidad")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"Malignant mean / Media maligna: {col[y == 0].mean():.3f}")
    print(f"Benign mean / Media benigna: {col[y == 1].mean():.3f}")
    print("EN: overlap matters; one feature alone is not a diagnosis.")
    print("ES: la superposición importa; una sola característica no constituye un diagnóstico.")

comparison_output = widgets.interactive_output(
    compare_groups,
    {"feature_name": comparison_feature},
)

display(widgets.VBox([comparison_feature, comparison_output]))

### One vector, every row / Un vector, todas las filas

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Broadcasting, drawn. A vector of five numbers lines up with the last axis and is reused down every one of the twelve rows. NumPy never copies it — the picture shows twelve subtractions, the memory holds one vector.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-03-broadcast.gif" alt="An animation showing a tensor of shape 3 by 4 by 5, then a vector of shape 5, then the vector lining up with the last axis of every row, then the result of the subtraction. The vector is never duplicated." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El broadcasting, dibujado. Un vector de cinco números se alinea con el último eje y se reutiliza en cada una de las doce filas. NumPy nunca lo copia: la imagen muestra doce restas, la memoria guarda un solo vector.</div>

### A scalar stretches everywhere / Un escalar se estira en todas direcciones

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

A scalar has no axis to line up with, so it is reused across every entry. The last frame is the trap the checkpoint sets: a bare `(4,)` meant as one offset per row does not broadcast, and `b[:, None]` makes it the `(4, 1)` column that does.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-03-scalar.gif" alt="An animation showing a 4 by 5 matrix minus the scalar 3, with the scalar drawn as twenty dashed copies, then the result, then a 4 by 1 column stretched across the rows as dashed copies." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un escalar no tiene ningún eje con el que alinearse, así que se reutiliza en cada entrada. El último fotograma es la trampa del punto de control: un <code>(4,)</code> pensado como un desplazamiento por fila no hace broadcasting, y <code>b[:, None]</code> lo convierte en la columna <code>(4, 1)</code> que sí lo hace.</div>

In [ ]:
#@title ⏸️ Step through the animations / Recorre las animaciones { display-mode: 'form' }

# Plumbing, not a lesson. The two animations above loop and then stop, and a
# GIF cannot be paused — so this fetches the same frames and hands them over
# one at a time, at whatever pace you read at.
# Plomería, no una lección: trae los mismos fotogramas y los entrega de uno en
# uno, al ritmo al que leas.

import io
import urllib.request

import ipywidgets as widgets
from IPython.display import display
from PIL import Image

gif_urls = [
    "https://project-delphi.github.io/tensors-workshop/images/cube-03-broadcast.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-03-scalar.gif",
]


def gif_frames(url):
    """Every frame of an animated GIF, as PNG bytes."""
    with urllib.request.urlopen(url, timeout=30) as response:
        gif = Image.open(io.BytesIO(response.read()))
    out = []
    try:
        while True:
            buffer = io.BytesIO()
            gif.convert("RGB").save(buffer, format="PNG")
            out.append(buffer.getvalue())
            gif.seek(gif.tell() + 1)
    except EOFError:
        pass
    return out


try:
    gif_cache = {url: gif_frames(url) for url in gif_urls}
except Exception as error:  # offline, or the site is down
    print("EN: could not reach the site, so there are no frames to step "
          "through.", error)
    print("ES: no se pudo acceder al sitio, así que no hay fotogramas que "
          "recorrer.", error)
else:
    gif_pick = widgets.Dropdown(
        options=[(url.rsplit("/", 1)[1], url) for url in gif_urls],
        description="Animation / Animación:",
        style={"description_width": "180px"},
    )
    gif_step = widgets.IntSlider(
        min=1, max=len(gif_cache[gif_urls[0]]), value=1,
        description="Frame / Fotograma:",
        style={"description_width": "180px"},
        continuous_update=False,
    )
    gif_prev = widgets.Button(description="◀ Prev")
    gif_next = widgets.Button(description="Next ▶")
    # An Image widget, deliberately, and never widgets.Output: a payload
    # leaving an Output widget makes nbclient wait out the whole cell timeout
    # (see scripts/test_notebooks.py). This one is a plain bytes trait.
    gif_view = widgets.Image(format="png",
                             layout=widgets.Layout(max_width="100%"))

    def gif_show(*_):
        frames = gif_cache[gif_pick.value]
        gif_step.max = len(frames)
        gif_view.value = frames[min(gif_step.value, len(frames)) - 1]

    def gif_bump(delta):
        def click(_):
            frames = gif_cache[gif_pick.value]
            gif_step.value = (gif_step.value - 1 + delta) % len(frames) + 1
        return click

    gif_prev.on_click(gif_bump(-1))
    gif_next.on_click(gif_bump(+1))
    gif_pick.observe(gif_show, names="value")
    gif_step.observe(gif_show, names="value")
    gif_show()

    display(widgets.VBox([
        gif_pick,
        widgets.HBox([gif_prev, gif_step, gif_next]),
        gif_view,
    ]))


### Broadcasting shape explorer / Explorador de formas de broadcasting

One question matters: **can a `(64,)` vector line up with the last axis of a
`(1797, 64)` matrix?**

Step through the subtraction and the division.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La pregunta que importa: ¿puede un vector <code>(64,)</code> alinearse con el último eje de una matriz <code>(1797, 64)</code>? Recorre la resta y la división.</div>

In [ ]:
#@title 🔍 Broadcasting explorer / Explorador de broadcasting — run me / ejecútame { display-mode: 'form' }

broadcast_step = widgets.ToggleButtons(
    options=[
        ("Original / Original", "original"),
        ("Subtract mean / Restar media", "center"),
        ("Divide by std / Dividir por std", "standardize"),
    ],
    value="original",
    description="Step / Paso:",
    style={"description_width": "100px"},
)

mean = D.mean(axis=0)
std = D.std(axis=0)
zero_variance = std == 0
safe_std = np.where(std == 0, 1.0, std)

def show_broadcast_step(step):
    if step == "original":
        arr = D
        en = "raw pixel values"
        es = "valores de píxel originales"
    elif step == "center":
        arr = D - mean
        en = "the 64 means are subtracted from every image row"
        es = "las 64 medias se restan de cada fila de imagen"
    else:
        arr = (D - mean) / safe_std
        en = "each centered pixel column is divided by its standard deviation"
        es = "cada columna de píxel centrada se divide por su desviación estándar"

    print("Input matrix / Matriz:", D.shape)
    print("mean.shape / forma de media:", mean.shape)
    print("std.shape / forma de std:", std.shape)
    print("Output / Salida:", arr.shape)
    print("EN:", en)
    print("ES:", es)

broadcast_output = widgets.interactive_output(
    show_broadcast_step,
    {"step": broadcast_step},
)

display(widgets.VBox([broadcast_step, broadcast_output]))

### Zero-variance explorer / Explorador de varianza cero

Pick a pixel position from `0` to `63`. You get its row and column in the
`8 × 8` image, its mean, its standard deviation, whether it is constant, and
every value it takes across all 1,797 images.

Try a highlighted zero-variance pixel, then one near the centre of the digit.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige una posición de píxel entre <code>0</code> y <code>63</code>: fila y columna, media, desviación estándar, si es constante, y todos sus valores en las 1.797 imágenes. Prueba un píxel de varianza cero y después uno del centro del dígito.</div>

## Predict first / Predice primero

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Someone says the following. **Decide whether they are right before you
reveal anything** — commit to one answer, then open the check.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,0.1);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">THE CLAIM · LA AFIRMACIÓN</div><div style="margin:.55em 0">&ldquo;Standardizing produced <code>NaN</code>, so <b>the standardization code is wrong</b>.&rdquo;</div></div>

A prediction you have committed to is worth more than one you keep
adjusting as the answer appears.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Alguien afirma que, como la estandarización produjo <code>NaN</code>, <b>el código está mal</b>. Decide si tiene razón <b>antes</b> de revelar la comprobación.</div>

In [ ]:
#@title 🤔 Predict: is that NaN a bug in your code? / Predice: ¿ese NaN es un error de tu código? — run me / ejecútame { display-mode: 'form' }

# --- counterexample / contraejemplo (tested in tests/test_teaching_materials.py) ---
import numpy as np

# Column 0 never changes across the samples, exactly like the three digit
# pixels that are constant in all 1,797 images.
pred_D = np.array([[0., 1., 5.], [0., 3., 9.], [0., 2., 7.]])
pred_mean = pred_D.mean(axis=0)
pred_std = pred_D.std(axis=0)

assert pred_std[0] == 0.0
with np.errstate(invalid="ignore", divide="ignore"):
    pred_Z = (pred_D - pred_mean) / pred_std

assert np.isnan(pred_Z[:, 0]).all()
assert np.isfinite(pred_Z[:, 1:]).all()
# --- end counterexample / fin del contraejemplo ---

import ipywidgets as widgets
from IPython.display import display

# --- how the question is laid out / cómo se presenta la pregunta ---
# Radio buttons rather than a dropdown. Four bilingual answers squeezed into
# one 640px line were hard to read, and a dropdown hides three of them until
# you open it -- the wrong shape for a question whose whole point is weighing
# the options against each other. One per line, with room around them.
# Botones de opción en vez de un desplegable: una respuesta por línea.
import contextlib
import html as pred_html
import io

PRED_ACCENT = "#7c3aed"
PRED_SANS = "ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"
PRED_MONO = "ui-monospace,SFMono-Regular,Menlo,Consolas,monospace"


def pred_tag(text):
    """A small EN / ES marker, in words rather than in colour alone."""
    return (f'<span style="font:700 10px/1 {PRED_MONO};letter-spacing:.16em;'
            f'color:{PRED_ACCENT};opacity:.8;margin-right:9px;'
            f'vertical-align:.12em">{text}</span>')


def pred_is_measurement(line):
    """True for a printed reading, false for a sentence.

    A reading wants monospace and tight rows so the numbers line up under one
    another; a sentence wants prose type and room. The two used to share one
    13px monospace column, which is most of why the reveal read as a wall.
    """
    if ":" not in line:
        return False
    tail = line.rsplit(":", 1)[1].strip()
    return bool(tail) and (tail[0].isdigit()
                           or tail[0] in "[(-+."
                           or tail.startswith(("True", "False", "nan", "inf")))


def pred_panel(text):
    """The reveal, laid out instead of printed.

    Exactly the same words: `check_prediction` still prints, and this catches
    what it printed and gives it typography. EN and ES stay written out as
    tags rather than becoming a colour, because a reader who cannot see the
    colour still has to be able to tell the two apart.
    """
    blocks = []
    for line in text.rstrip("\n").split("\n"):
        stripped = line.strip()
        if not stripped:
            blocks.append('<div style="height:12px"></div>')
        elif stripped.startswith(("EN:", "ES:")):
            tag, body = stripped[:2], stripped[3:].strip()
            blocks.append(
                f'<p style="margin:.55em 0;font:400 15px/1.8 {PRED_SANS}">'
                f'{pred_tag(tag)}{pred_html.escape(body)}</p>')
        elif pred_is_measurement(stripped):
            blocks.append(
                f'<div style="font:400 13.5px/2.0 {PRED_MONO};'
                f'white-space:pre-wrap">{pred_html.escape(stripped)}</div>')
        else:
            blocks.append(
                f'<p style="margin:.55em 0;font:600 15.5px/1.75 {PRED_SANS}">'
                f'{pred_html.escape(stripped)}</p>')
    return (f'<div style="border-left:4px solid {PRED_ACCENT};'
            f'background:rgba(130,130,150,.08);border-radius:0 10px 10px 0;'
            f'padding:16px 20px;margin:.4em 0 0">{"".join(blocks)}</div>')


def pred_render(choice, reveal):
    """Run the check, catch what it prints, and show it laid out."""
    caught = io.StringIO()
    with contextlib.redirect_stdout(caught):
        check_prediction(choice, reveal)
    display(widgets.HTML(pred_panel(caught.getvalue())))

pred_choice = widgets.RadioButtons(
    options=[
        ("— choose one / elige una —", None),
        ("Right — NaN means the code is broken / Correcto — NaN significa código roto", "code_bug"),
        ("Wrong — a constant column has std 0 / Incorrecto — una columna constante tiene std 0", "data_property"),
        ("Wrong — NumPy always returns 0 there / Incorrecto — NumPy siempre devuelve 0 ahí", "numpy_zero"),
    ],
    value=None,
    description="",
    layout=widgets.Layout(width="auto", margin="0 0 6px 0"),
)

pred_reveal = widgets.Checkbox(
    value=False,
    description="Show me the answer / Muéstrame la respuesta",
    indent=False,
    layout=widgets.Layout(margin="10px 0 4px 0"),
)

def check_prediction(choice, reveal):
    if choice is None:
        print("Choose an answer first / Elige una respuesta primero.")
        return

    if not reveal:
        print("Answer saved / Respuesta guardada.")
        print("Tick the box above when you are ready / Marca la casilla de arriba\n      cuando quieras.".replace("\n      ", " "))
        return

    print("Column means / Medias por columna:", pred_mean)
    print("Column stds / Desviaciones:      ", pred_std)
    print()
    print("Column 0 is constant / La columna 0 es constante:", bool(pred_std[0] == 0))
    print("NaN in column 0 / NaN en la columna 0:", bool(np.isnan(pred_Z[:, 0]).all()))
    print("Other columns finite / Otras columnas finitas:", bool(np.isfinite(pred_Z[:, 1:]).all()))
    print()
    if choice == "data_property":
        print("You were right / Acertaste.")
    else:
        print("You were wrong — read on / Te equivocaste; sigue leyendo.")
    print()
    print("EN: the formula divided by zero because that column never varies. The code is correct; it reported a property of the data. A zero-variance feature is a finding, not a bug.")
    print("ES: la fórmula dividió entre cero porque esa columna nunca varía. El código es correcto: informó de una propiedad de los datos. Una característica de varianza cero es un hallazgo, no un error.")

# The one <style> block in these notebooks, and the markdown rule does not
# cover it. ipywidgets gives no way to set the space between radio options
# from Python, and this is *widget output*, not a markdown cell: Colab strips
# <style> from markdown -- which is why every box in these notebooks is
# inline-styled -- but renders it in an output, the same path pandas' own
# Styler uses. Scoped to one added class so it can reach nothing else, and if
# it is ever dropped the options still work, just closer together.
pred_choice.add_class("pred-radio")

display(widgets.HTML(
    "<style>"
    ".pred-radio .widget-radio-box label{display:flex;align-items:flex-start;"
    "margin:0 0 13px;font:400 15px/1.6 " + PRED_SANS + "}"
    ".pred-radio input[type=radio]{flex:none;margin:4px 11px 0 0;"
    "transform:scale(1.15)}"
    "</style>"
))

pred_output = widgets.interactive_output(
    pred_render,
    {"choice": pred_choice, "reveal": pred_reveal},
)

pred_heading = widgets.HTML(
    f'<div style="font:700 11px/1.6 {PRED_MONO};letter-spacing:.18em;'
    f'color:{PRED_ACCENT};margin:2px 0 12px">'
    f'YOUR PREDICTION \u00b7 TU PREDICCI\u00d3N</div>'
)

display(widgets.VBox(
    [pred_heading, pred_choice, pred_reveal, pred_output],
    layout=widgets.Layout(padding="2px 0 14px 0"),
))

In [ ]:
#@title 🔍 Zero-variance explorer / Explorador de varianza cero — run me / ejecútame { display-mode: 'form' }

zero_indices = np.flatnonzero(zero_variance)

pixel_slider = widgets.IntSlider(
    value=int(zero_indices[0]) if len(zero_indices) else 0,
    min=0,
    max=63,
    step=1,
    description="Pixel / Píxel:",
    continuous_update=False,
    style={"description_width": "100px"},
)

def explore_pixel(pixel):
    values = D[:, pixel]
    row, col = divmod(pixel, 8)

    plt.close("all")
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.2))

    axes[0].imshow(D.mean(axis=0).reshape(8, 8), cmap="gray")
    axes[0].scatter([col], [row], s=180, marker="s", facecolors="none")
    axes[0].set_title(f"Pixel {pixel} → ({row}, {col})")
    axes[0].axis("off")

    axes[1].hist(values, bins=20)
    axes[1].set_title("Values across images / Valores entre imágenes")
    axes[1].set_xlabel("pixel value / valor del píxel")
    axes[1].set_ylabel("count / cantidad")

    plt.tight_layout()
    plt.show()

    print("Mean / Media:", float(values.mean()))
    print("Std / Desviación estándar:", float(values.std()))
    print("Zero variance / Varianza cero:", bool(values.std() == 0))
    if values.std() == 0:
        print("EN: every image has exactly the same value at this pixel.")
        print("ES: todas las imágenes tienen exactamente el mismo valor en este píxel.")
    else:
        print("EN: this pixel changes across images and therefore carries variation.")
        print("ES: este píxel cambia entre imágenes y por eso contiene variación.")

pixel_output = widgets.interactive_output(
    explore_pixel,
    {"pixel": pixel_slider},
)

display(widgets.VBox([pixel_slider, pixel_output]))

## What just happened / Qué acaba de pasar

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Three ways to select, one way to transform, all on real data.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">FOUR IDEAS · CUATRO IDEAS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span><b>Index by meaning.</b> A feature name says what you meant; a column number does not.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span><b>Fancy indexing chooses positions.</b> Use it when you know which rows.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span><b>Boolean masks choose by condition.</b> <code>True</code> means “keep this one”.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span><b>Broadcasting exposes real-data traps.</b> Standardization failed at exactly the columns whose standard deviation was zero.</div></div>

Two findings, from these two datasets:

- `mean radius` is larger on average in the malignant group than in the benign
  group.
- Exactly **three pixel positions** are constant across all 1,797 digit images.

Both are properties of *these* datasets. Neither is a general rule.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>1 ·</b> indexa por significado; <b>2 ·</b> fancy indexing elige posiciones; <b>3 ·</b> las máscaras eligen por condición; <b>4 ·</b> el broadcasting destapa trampas de los datos reales.<br><br>Dos hallazgos <i>de estos conjuntos</i>: <code>mean radius</code> es mayor en promedio en el grupo maligno, y exactamente <b>tres posiciones de píxel</b> son constantes en las 1.797 imágenes. No son reglas generales.</div>

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

## Done with this section / Fin de esta sección

Next / Siguiente: **04 · Reshape and transpose real images / Reshape y transposición de imágenes reales** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/04-reshape-and-transpose.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)